# ISOMIGA AD coloc protective LINCS matching

This notebook summarizes preliminary drug-prioritization data for ISOMIGA microglia eQTL colocalizations with AD GWAS. The protective expression direction is inferred from the signs of the AD GWAS SNP beta and eQTL SNP beta: same sign means increased expression tracks AD risk, opposite sign means increased expression tracks protection.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from plotnine import *

DATA = Path("../data/processed")
FIG = Path("../results/figures")
FIG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 80)

## Protective expression targets

In [ ]:
targets = pd.read_csv(DATA / "protective_expression_targets.tsv", sep="\t")
gene_targets = pd.read_csv(DATA / "protective_expression_gene_summary.tsv", sep="\t")
coverage_path = DATA / "lincs_target_gene_coverage.tsv"
coverage = pd.read_csv(coverage_path, sep="\t") if coverage_path.exists() else None

print(f"High-H4 coloc target rows: {len(targets):,}")
print(f"Unique genes: {gene_targets['gene_name'].nunique():,}")
print(gene_targets['protective_direction_label'].value_counts(dropna=False))
gene_targets.head(20)

In [ ]:
p = (
    ggplot(gene_targets, aes("protective_direction_label", fill="protective_direction_label"))
    + geom_bar(show_legend=False)
    + labs(x="Protective expression direction", y="Colocalizing genes", title="ISOMIGA AD coloc target directions")
    + theme_bw()
    + theme(figure_size=(5, 3))
)
p.save(FIG / "target_direction_counts.png", dpi=300)
p

In [ ]:
top_targets = (
    gene_targets.sort_values(["max_h4", "abs_protective_score"], ascending=False)
    .head(25)
    .assign(gene_name=lambda x: pd.Categorical(x["gene_name"], categories=x["gene_name"][::-1], ordered=True))
)
p = (
    ggplot(top_targets, aes("max_h4", "gene_name", color="protective_direction_label", size="n_coloc"))
    + geom_point()
    + labs(x="Max coloc PPF.H4", y="Gene", color="Protective direction", size="Coloc rows", title="Top ISOMIGA AD coloc eQTL targets")
    + theme_bw()
    + theme(figure_size=(7, 6))
)
p.save(FIG / "top_coloc_targets.png", dpi=300)
p

## LINCS target coverage

In [ ]:
if coverage is None:
    print("Run scripts/extract_lincs_thp1_targets.py to produce LINCS target coverage.")
else:
    display(coverage['in_lincs'].value_counts(dropna=False))
    display(coverage.sort_values(['in_lincs', 'gene_name'], ascending=[True, True]).head(30))

## THP1 drug scores

In [ ]:
drug_scores_path = DATA / "lincs_thp1_protective_drug_scores.tsv"
drug_gene_path = DATA / "lincs_thp1_protective_drug_gene_scores.tsv"
if drug_scores_path.exists() and drug_gene_path.exists():
    drug_scores = pd.read_csv(drug_scores_path, sep="\t")
    drug_gene = pd.read_csv(drug_gene_path, sep="\t")
    display(drug_scores.head(25))
else:
    drug_scores = None
    drug_gene = None
    print("Run scripts/run_pipeline.sh through LINCS extraction/scoring to populate drug score files.")

In [ ]:
if drug_scores is not None:
    top_drugs = (
        drug_scores.head(20)
        .assign(pert_iname=lambda x: pd.Categorical(x["pert_iname"], categories=x["pert_iname"][::-1], ordered=True))
    )
    p = (
        ggplot(top_drugs, aes("weighted_mean_protective_push_z", "pert_iname"))
        + geom_col(fill="#3b6ea8")
        + geom_point(aes(size="n_protective_genes"), color="#111111")
        + labs(x="Weighted mean protective push z", y="Compound", size="Protective genes", title="Top THP1 LINCS compounds by protective expression push")
        + theme_bw()
        + theme(figure_size=(7, 6))
    )
    p.save(FIG / "top_lincs_protective_drugs.png", dpi=300)
    p

In [ ]:
if drug_gene is not None and drug_scores is not None:
    selected = drug_scores.head(12)[["pert_id", "pert_iname"]]
    heat = drug_gene.merge(selected, on=["pert_id", "pert_iname"])
    gene_order = (
        heat.groupby("gene_name")["max_h4"].max().sort_values(ascending=False).index.tolist()
    )
    heat["gene_name"] = pd.Categorical(heat["gene_name"], categories=gene_order[::-1], ordered=True)
    heat["pert_iname"] = pd.Categorical(heat["pert_iname"], categories=selected["pert_iname"][::-1], ordered=True)
    p = (
        ggplot(heat, aes("pert_iname", "gene_name", fill="protective_push_z"))
        + geom_tile(color="white")
        + scale_fill_gradient2(low="#9c2f2f", mid="#f7f7f7", high="#2f7d4f", midpoint=0)
        + labs(x="Compound", y="Target gene", fill="Protective push z", title="Per-gene protective push for top compounds")
        + theme_bw()
        + theme(figure_size=(8, max(4, 0.25 * len(gene_order))), axis_text_x=element_text(rotation=45, ha="right"))
    )
    p.save(FIG / "top_drug_gene_protective_push_heatmap.png", dpi=300)
    p

## Export prelim tables

In [ ]:
if drug_scores is not None:
    cols = ["pert_iname", "n_target_genes", "n_protective_genes", "n_strong_protective_genes", "weighted_mean_protective_push_z", "fraction_genes_protective", "min_n_signatures"]
    prelim = drug_scores.loc[:, cols].head(50)
    prelim.to_csv(DATA / "prelim_top_lincs_thp1_protective_drugs.tsv", sep="\t", index=False)
    display(prelim.head(20))